# 00. 阅读地图：从文档用户到 Runtime 贡献者

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


## 本套指南解决什么问题

SGLang 官网文档非常适合“我该传什么参数、怎么启动服务”。但贡献者还需要另一张地图：

- 一个请求从 HTTP API 到 tokenizer、scheduler、model runner、detokenizer 的路径。
- 为什么 RadixAttention 不是一个普通 attention kernel，而是“prefix KV cache + paged attention + scheduler”的协同设计。
- Structured Outputs 里 JSON schema / regex / EBNF 如何变成每步采样前的 token mask，以及为什么文档中会提到 compressed FSM / jump-forward。
- Speculative decoding 如何把 draft、verify、accept/reject、KV 提交串起来。
- HiCache 如何把 RadixAttention 的 L1 GPU KV cache 扩展成 L1/L2/L3 分层缓存。
- 新增特性通常落在哪些扩展点：server args、scheduler、model runner、attention backend、memory pool、grammar backend、spec worker、storage backend。

这组 notebook 的目标不是替代 API 文档，而是把“文档中的 Feature 名词”连回“源码中的对象、状态和边界”。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 推荐阅读顺序

1. `00_reading_map.ipynb`：总览与源码地图。
2. `01_runtime_request_path.ipynb`：请求路径和进程/manager 边界。
3. `02_radix_attention_prefix_cache.ipynb`：RadixAttention、RadixCache、prefix reuse。
4. `03_scheduler_memory_execution.ipynb`：scheduler 如何把请求变成 GPU forward batch。
5. `04_structured_outputs_fsm.ipynb`：grammar backend、token mask、jump-forward / compressed FSM。
6. `05_speculative_decoding.ipynb`：EAGLE、STANDALONE、NGRAM、DFLASH 的共同骨架。
7. `06_hicache_disaggregation_new_features.ipynb`：HiCache、PD/EPD disaggregation、新特性的集成模式。
8. `07_contributor_playbook.ipynb`：贡献者改代码的路线、测试入口和 PR 思维方式。


In [ ]:
docs_index = read_rel("docs/index.rst")
sections = re.findall(r":caption:\s*(.+?)\n\n(.*?)(?=\n\.\. toctree::|\Z)", docs_index, flags=re.S)
for caption, body in sections:
    entries = [line.strip() for line in body.splitlines() if line.strip() and not line.strip().startswith(":")]
    print(f"\n[{caption}]")
    for entry in entries[:12]:
        print(" -", entry)
    if len(entries) > 12:
        print(f" - ... {len(entries)-12} more")


## 关键源码区域

贡献者最常接触的是 `python/sglang/srt`。其中 SRT 是 SGLang Runtime：

- `entrypoints/`：HTTP / OpenAI / Ollama / Engine API 入口。
- `managers/`：tokenizer、scheduler、detokenizer 及调度数据结构。
- `model_executor/`：model runner、ForwardBatch、CUDA graph runner、forward context。
- `layers/`：RadixAttention、MoE、quantization、attention backend glue。
- `mem_cache/`：KV memory pool、RadixCache、HiRadixCache、HiCache storage backend。
- `constrained/`：structured outputs 的 grammar backend。
- `speculative/`：speculative decoding 的 worker、输入结构和 accept/reject kernel。
- `disaggregation/`：prefill/decode/encoder disaggregation。


In [ ]:
interesting_dirs = [
    "python/sglang/srt/entrypoints",
    "python/sglang/srt/managers",
    "python/sglang/srt/model_executor",
    "python/sglang/srt/layers",
    "python/sglang/srt/mem_cache",
    "python/sglang/srt/constrained",
    "python/sglang/srt/speculative",
    "python/sglang/srt/disaggregation",
]
for d in interesting_dirs:
    files = sorted((ROOT / d).glob("*.py"))
    print(f"{d}: {len(files)} top-level .py files")
    for f in files[:8]:
        print("  ", f.name)


## Feature 到源码的索引

下面这个小表是后续章节的“找路牌”。你可以把它当成 grep 前的心智模型。


In [ ]:
feature_map = {
    "HTTP/OpenAI API": [
        "python/sglang/srt/entrypoints/http_server.py",
        "python/sglang/srt/entrypoints/openai/",
        "python/sglang/srt/managers/tokenizer_manager.py",
    ],
    "Scheduler / continuous batching": [
        "python/sglang/srt/managers/scheduler.py",
        "python/sglang/srt/managers/schedule_batch.py",
        "python/sglang/srt/managers/schedule_policy.py",
    ],
    "RadixAttention / prefix cache": [
        "python/sglang/srt/layers/radix_attention.py",
        "python/sglang/srt/mem_cache/radix_cache.py",
        "python/sglang/srt/mem_cache/base_prefix_cache.py",
    ],
    "Structured outputs / FSM": [
        "python/sglang/srt/constrained/base_grammar_backend.py",
        "python/sglang/srt/constrained/grammar_manager.py",
        "python/sglang/srt/constrained/xgrammar_backend.py",
        "python/sglang/srt/constrained/outlines_jump_forward.py",
    ],
    "Speculative decoding": [
        "python/sglang/srt/speculative/spec_info.py",
        "python/sglang/srt/speculative/eagle_worker_v2.py",
        "python/sglang/srt/speculative/ngram_worker.py",
    ],
    "HiCache": [
        "python/sglang/srt/mem_cache/hiradix_cache.py",
        "python/sglang/srt/managers/cache_controller.py",
        "python/sglang/srt/mem_cache/hicache_storage.py",
    ],
}
for k, paths in feature_map.items():
    print(f"\n{k}")
    for p in paths:
        print(" -", p)


## 读源码时的三个不变量

- **请求状态分层**：API 层看到字符串和 JSON；scheduler 层看到 `Req` / `ScheduleBatch`；model runner 层看到 GPU tensor 和 `ForwardBatch`。
- **KV cache 是核心资源**：吞吐并不只取决于 kernel，还取决于哪些 token 已经有 KV、哪些 KV 被锁住、哪些可以驱逐。
- **Feature 很少孤立存在**：例如 structured outputs 会影响 sampling，spec decoding 会影响 KV 分配，HiCache 会影响 prefix matching 和 prefill 调度。
